# 02 - MLP Model Training

Run Notebook 01 first. This notebook performs one main model-training task in each numbered step.

## Step 1 - Import the libraries

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

## Step 2 - Set the file paths

In [ ]:
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_FILE = PROJECT / 'data' / 'processed_data.npz'
MODELS_DIR = PROJECT / 'models'
MODEL_FILE = MODELS_DIR / 'mlp_model.pt'
MODELS_DIR.mkdir(exist_ok=True)

## Step 3 - Set the random seed

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

## Step 4 - Select the training device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training device:', device)

## Step 5 - Load the processed dataset

In [ ]:
if not PROCESSED_FILE.exists():
    raise FileNotFoundError('Run Notebook 01 before training the model.')

processed_data = np.load(PROCESSED_FILE)
X_train = processed_data['X_train']
X_test = processed_data['X_test']
y_train = processed_data['y_train']
y_test = processed_data['y_test']

## Step 6 - Load the class names

In [ ]:
label_encoder = joblib.load(MODELS_DIR / 'label_encoder.joblib')
class_names = label_encoder.classes_
number_of_classes = len(class_names)
print('Classes:', list(class_names))

## Step 7 - Check the dataset shapes

In [ ]:
print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

## Step 8 - Convert the arrays to PyTorch tensors

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
y_train_tensor = torch.from_numpy(y_train)
X_test_tensor = torch.from_numpy(X_test)
y_test_tensor = torch.from_numpy(y_test)

## Step 9 - Create the PyTorch datasets

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

## Step 10 - Create the data loaders

In [ ]:
BATCH_SIZE = 1024
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Step 11 - Define the MLP architecture

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_size),
        )

    def forward(self, features):
        return self.layers(features)

## Step 12 - Create the MLP model

In [ ]:
input_size = X_train.shape[1]
model = MLP(input_size, number_of_classes).to(device)
model

## Step 13 - Calculate the class weights

In [ ]:
class_counts = np.bincount(y_train, minlength=number_of_classes)
weight_values = len(y_train) / (number_of_classes * class_counts)
class_weights = torch.tensor(weight_values, dtype=torch.float32, device=device)
print('Class weights:', class_weights)

## Step 14 - Create the loss function

In [ ]:
loss_function = nn.CrossEntropyLoss(weight=class_weights)
loss_function

## Step 15 - Create the optimizer

In [ ]:
LEARNING_RATE = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
optimizer

## Step 16 - Train the model

In [ ]:
EPOCHS = 15
training_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)

    average_loss = total_loss / len(train_dataset)
    training_losses.append(average_loss)
    print(f'Epoch {epoch + 1:02d}/{EPOCHS} - Loss: {average_loss:.4f}')

## Step 17 - Display the training loss

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), training_losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('MLP training loss')
plt.grid(alpha=0.3)
plt.show()

## Step 18 - Make predictions using the test set

In [ ]:
model.eval()
test_predictions = []
test_targets = []

with torch.no_grad():
    for features, labels in test_loader:
        outputs = model(features.to(device))
        predicted_classes = outputs.argmax(dim=1).cpu().numpy()
        test_predictions.extend(predicted_classes)
        test_targets.extend(labels.numpy())

## Step 19 - Calculate the evaluation metrics

In [ ]:
accuracy = accuracy_score(test_targets, test_predictions)
macro_f1 = f1_score(test_targets, test_predictions, average='macro')

print(f'Test accuracy: {accuracy:.4f}')
print(f'Test macro-F1: {macro_f1:.4f}')
print(classification_report(
    test_targets,
    test_predictions,
    target_names=class_names,
    zero_division=0,
))

## Step 20 - Display the confusion matrix

In [ ]:
matrix = confusion_matrix(test_targets, test_predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(
    matrix,
    cmap='Blues',
    fmt='d',
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel('Predicted class')
plt.ylabel('True class')
plt.title('MLP confusion matrix')
plt.tight_layout()
plt.show()

## Step 21 - Save the trained model

In [ ]:
torch.save(model.state_dict(), MODEL_FILE)
print('Model saved:', MODEL_FILE)